# AI AINAK V2 — Notebook 07: V1 Baseline vs. V2 Capability Comparison

## 1. Executive Summary
This notebook delivers a rigorous, data-driven comparison between **AI AINAK Version 1** and the **Version 2 Capability Expansion**.

All performance figures, latencies, and accuracy numbers are drawn directly from the **independent capability audit** conducted on the host machine (Intel Core i5-11400H @ 2.70 GHz, 12 logical cores, `CUDA=False`).

---

### Core Dimension Summary
| Capability Dimension | V1 Baseline | V2 Expanded | Evaluation Substrate |
| :--- | :--- | :--- | :--- |
| **Gesture Classes** | 4 classes | **25 classes** (+525% expansion) | 32 physical vs 200 synthetic sequences |
| **Object Categories** | 80 COCO classes | **251 open-vocab categories** (+213% expansion) | Walkthrough recordings (29 vs 54 detections) |
| **Hazard Intelligence** | 3 heuristic bands | **4 standardized tiers** & 8 canonical scenarios | Dynamic intent (parked vs looming car, TTC) |
| **Audio Recognition** | 50 ESC-50 classes | **50 ESC-50 classes** (preserved) | PANNs CNN14 + Downstream MLP (89.75% acc) |
| **Transformer Fusion** | Validated contract | **Semantic drift under modulo-80** | Adapter required before V2 deployment |
| **End-to-End Latency** | **68.5 ms (14.6 FPS)** | **171.2 ms (5.8 FPS)** | Intel Core i5-11400H CPU-only benchmark |

In [ ]:
# Step 1: Install Dependencies
!pip install -q matplotlib numpy

In [ ]:
# Step 2: Environment & Asset Discovery
import os, sys, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    try: drive.mount("/content/drive")
    except: pass
    PROJECT_ROOT = Path("/content/drive/MyDrive/AINAK_V2_Colab")
    if not PROJECT_ROOT.exists(): PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path.cwd().resolve()
    while not (PROJECT_ROOT / "artifacts").is_dir() and PROJECT_ROOT.parent != PROJECT_ROOT:
        PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project Root: {PROJECT_ROOT}")

In [ ]:
# Step 3: Measured Latency & Throughput Benchmark
# Empirically measured CPU benchmarks on Intel Core i5-11400H (12 logical cores)
modules = ["Vision", "Gesture", "Audio", "Fusion", "End-to-End"]
v1_latencies = [62.82, 1.33, 62.40, 2.82, 68.50]
v2_latencies = [174.92, 1.23, 62.40, 2.82, 171.21]

x = np.arange(len(modules))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5), dpi=130)
rects1 = ax.bar(x - width/2, v1_latencies, width, label="V1 Baseline", color="#1976D2")
rects2 = ax.bar(x + width/2, v2_latencies, width, label="V2 Expanded", color="#FF7043")

ax.set_ylabel("Inference Latency (ms, CPU)", fontsize=11)
ax.set_title("Inference Latency Benchmark (Intel Core i5-11400H @ 2.70 GHz)\nV1 End-to-End: 68.5 ms (14.6 FPS) vs V2: 171.2 ms (5.8 FPS)", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(modules, fontsize=10)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

for rect in rects1:
    h = rect.get_height()
    ax.annotate(f"{h:.1f}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)
for rect in rects2:
    h = rect.get_height()
    ax.annotate(f"{h:.1f}", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8, fontweight="bold")

fig.tight_layout()
plt.show()

In [ ]:
# Step 4: Display Precomputed Comparison Visualizations
fig_comp = PROJECT_ROOT / "notebooks" / "v2_colab" / "assets" / "presentation_figures" / "executive_readiness_scorecard.png"
fig_arch = PROJECT_ROOT / "notebooks" / "v2_colab" / "assets" / "architecture" / "v2_system_architecture.png"

if fig_comp.exists() and fig_arch.exists():
    from PIL import Image
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=120)
    axes[0].imshow(Image.open(fig_arch))
    axes[0].axis("off")
    axes[0].set_title("Unified Multimodal Dual Pipeline Architecture", fontsize=11)
    
    axes[1].imshow(Image.open(fig_comp))
    axes[1].axis("off")
    axes[1].set_title("Executive Readiness Scorecard & Audit Verdicts", fontsize=11)
    plt.tight_layout()
    plt.show()

### 5. Detailed Scientific Disclosures for College Faculty Presentation

#### A. Gesture Recognition (25 Classes)
- **Synthetic Kinematics**: The 1,200 sequences were synthesized via parametric kinematic curves (`generate_base_kinematics()`). The 99.24% test-train correlation accounts for the 100% test accuracy.
- **Occlusion Collapse**: Dropping hand/wrist landmarks drops accuracy to **4.5%**. Non-frontal yaw ($>30^\circ$) drops accuracy to **36%–52%**.
- **OOD Rejection**: 100% rejection on Gaussian noise, but **12.0% false trigger rate on unfamiliar natural human gestures** at $\tau = 0.70$.

#### B. Object Detection (251 Categories)
- **Open-Vocabulary Neck**: YOLO-World uses frozen CLIP text embeddings, not 251 supervised detector heads (63.3% rely on open-vocab zero-shot prompts).
- **Detection Density**: Yields a **1.86x increase** (54 vs 29 detections) on physical walkthrough recordings.
- **Synonym Overlaps**: Generates duplicate boxes on semantic synonyms (`pavement`/`sidewalk`).

#### C. Multimodal Transformer Fusion
- **Semantic Drift**: Naive modulo-80 indexing causes severe feature misalignment (cosine similarity 0.5894, 100% false alarm rate on informational sessions).
- **Remediation**: A learned 2-layer linear projection adapter ($251 \to 128$) is required before zero-shot V2 deployment.

---

### Conclusion
**AINAK V2** represents a substantial capability expansion, broadening real-world assistive perception while honestly identifying the engineering boundaries (occlusion sensitivity, open-vocabulary synonym suppression, and linear fusion adapter training) required for certified commercial deployment.

In [ ]:
# Step 6: Export Comparison Summary
out_dir = PROJECT_ROOT / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "v1_v2_comparison_metrics.json", "w", encoding="utf-8") as f:
    json.dump({
        "comparison": {
            "gesture_classes": {"v1": 4, "v2": 25},
            "vision_categories": {"v1": 80, "v2": 251},
            "hazard_tiers": {"v1": 3, "v2": 4},
            "end_to_end_latency_ms": {"v1": 68.50, "v2": 171.21},
            "fps_throughput": {"v1": 14.6, "v2": 5.8},
            "transformer_readiness": {"v1": "Validated", "v2": "Requires Learned Adapter"}
        }
    }, f, indent=2)

print("Comparison summary successfully exported!")